## This will be used to explore data.
### It will access each bird species in terms of:
- Recording count
- Average quality
- Count of recording qualities
- Average length of recordings


In [94]:
import json
import pandas as pd


# Load bird species list and convert to DataFrame
with open("../../species_list_small.json", "r") as f:
    bird_list = json.load(f)

rankings = bird_list.keys()
birds = bird_list.values()
birds_df = pd.DataFrame(birds, index=rankings)
birds_df

,common_name,sci_name
1,European Robin,Erithacus rubecula
2,Eurasian Blackbird,Turdus merula
3,Eurasian Wren,Troglodytes troglodytes
4,Eurasian Blue Tit,Cyanistes caeruleus
5,Great Tit,Parus major
6,European Goldfinch,Carduelis carduelis
7,Common Chaffinch,Fringilla coelebs
8,Song Thrush,Turdus philomelos
9,Eurasian Magpie,Pica pica
10,Eurasian Jackdaw,Corvus monedula


In [95]:
## Load API keys
from dotenv import load_dotenv
import os

load_dotenv()
XENO_CANTO_API_KEY = os.environ.get('XENO_CANTO_API_KEY')
print("Xeno-Canto API Key loaded:", XENO_CANTO_API_KEY)

Xeno-Canto API Key loaded: d15cad7bdbfb958edb8a549a81aae802462ae7fd


In [96]:
import time
import requests


def _get_with_retry(url, params, max_retries=3):
    last_exception = None

    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params=params)
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            last_exception = e
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
    raise last_exception

In [97]:
BASE_URL = "https://xeno-canto.org/api/3/recordings"

def get_recordings_data_for_species(sci_name, per_page=500):
    all_recordings = []
    page = 1

    while True:
        params = {
            "query": f'sp:"{sci_name}"',
            "page": page,
            "per_page": per_page,
            "key": XENO_CANTO_API_KEY
        }

        data = _get_with_retry(BASE_URL, params)
        all_recordings.extend(data["recordings"])

        if page >= data.get("numPages", 1):
            break
        page += 1
    return all_recordings

#### For each species recordings list:
- Total count
- Total european count
- Average recording length
- Location counts (object)
- Quality counts (object)



In [98]:
# Simple list of European countries for filtering
europe_countries = [
    "United Kingdom", "Ireland", "France", "Germany", "Spain", "Portugal",
    "Italy", "Netherlands", "Belgium", "Switzerland", "Austria", "Denmark",
    "Sweden", "Norway", "Finland", "Poland", "Czech Republic", "Slovakia",
    "Hungary", "Greece", "Romania", "Bulgaria", "Croatia", "Slovenia",
    "Estonia", "Latvia", "Lithuania", "Luxembourg", "Liechtenstein",
    "Iceland", "Malta"
]


In [99]:
def _xeno_length_to_seconds(s):
    if not s or ":" not in s:
        return None
    
    parts = s.split(":")

    parts = list(map(int, parts))
    if len(parts) == 2:  # MM:SS
        m, sec = parts
        return m*60 + sec
    elif len(parts) == 3:  # H:MM:SS (rare)
        h, m, sec = parts
        return h*3600 + m*60 + sec

In [100]:
from collections import Counter

# Need to iterate through the pandas DataFrame and collect stats for each species
species_stats = {}

for bird in birds_df.itertuples():
    sci_name = bird.sci_name
    common_name = bird.common_name
    recordings = get_recordings_data_for_species(sci_name) ## get all recordings for species

    total_count = len(recordings)
    european_count = sum(1 for rec in recordings if rec['cnt'] in europe_countries)
    lens = [r.get("length") for r in recordings]
    lengths = [_xeno_length_to_seconds(r.get("length")) for r in recordings]
    lengths = [x for x in lengths if x is not None]
    avg_length = (sum(lengths) / len(lengths)) if lengths else 0.0

    q_counts = Counter(r.get("q") for r in recordings if r.get("q"))
    quality_counts = {k: q_counts.get(k, 0) for k in ["A", "B", "C", "D", "E"]}
    
    c_counts = Counter(r.get("cnt") for r in recordings if r.get("cnt"))
    country_counts = dict(sorted(c_counts.items(), key=lambda item: item[1], reverse=True)[:10]) ## limit to top 10 countries

    species_stats[sci_name] = {
        "total_count": total_count,
        "european_count": european_count,
        "avg_length": avg_length,
        "quality_counts": quality_counts,
        "country_counts": country_counts
    }

    print(f"{common_name} ({sci_name}): {species_stats[sci_name]}")
    print("Lengths:", lens)




European Robin (Erithacus rubecula): {'total_count': 7315, 'european_count': 6931, 'avg_length': 140.2952836637047, 'quality_counts': {'A': 898, 'B': 3341, 'C': 2479, 'D': 456, 'E': 141}, 'country_counts': {'France': 1368, 'United Kingdom': 1219, 'Germany': 816, 'Spain': 740, 'Portugal': 510, 'Poland': 416, 'Netherlands': 392, 'Sweden': 282, 'Ireland': 256, 'Italy': 239}}
Lengths: ['0:08', '1:18', '0:21', '2:41', '0:38', '0:57', '2:49', '0:31', '0:22', '0:18', '3:37', '2:48', '1:55', '0:32', '0:18', '0:05', '0:04', '3:40', '1:57', '0:35', '0:20', '2:14', '1:24', '0:30', '0:25', '4:09', '1:16', '2:00', '4:43', '4:47', '1:44', '1:32', '0:36', '1:30', '0:52', '5:07', '1:27', '2:21', '1:21', '1:00', '16:13', '3:48', '1:01', '0:43', '3:37', '3:14', '0:41', '1:19', '1:17', '2:14', '5:03', '4:03', '1:50', '2:09', '2:41', '1:58', '5:30', '0:28', '2:14', '2:03', '3:08', '5:00', '1:10', '1:50', '3:23', '0:39', '0:41', '0:41', '0:44', '2:49', '2:08', '1:03', '0:43', '0:23', '1:14', '0:12', '0:39'